# Probabilistic cell typing

This notebooks guides you through the integration of ISS data with pre-existing clustered and annotated scRNAseq datasets, using Probabilistic Cell Typing (PCIseq).

Please have a look at:
https://www.nature.com/articles/s41592-019-0631-4
https://github.com/acycliq/pciSeq


Using this method, and if the genes measured by ISS have been accurately chosen for the task, it is possible to link these 2 modalities, and put on a geographical map the clusters inferred by scRNAseq in a coupled dataset.

## Import the necessary modules

In [ ]:
import ISS_postprocessing.pciseq as PCIseq


## Import the segmentation mask and the ISS data

In the following steps, we first load the main inputs required for building the ISS–scRNAseq integration dataset:

* **scRNAseq reference** (`sc_file`):  
  A gene-by-cell type (or cluster) expression matrix used as reference for probabilistic cell typing.

* **Segmentation mask** (`coo_file`):  
  A sparse label image (saved in `.npz` format) where each pixel is assigned to a cell ID. This defines the spatial boundaries of individual cells.

* **Decoded ISS spots** (`spots_file`):  
  A table (CSV) containing the coordinates and gene identities of all decoded spots. Each spot will later be assigned to a cell in the segmentation mask.


Together, these inputs are combined to construct a spatially resolved **gene-by-cell count matrix**.


### Read the single cell RNA sequencing dataset



In [ ]:
sc_file = '/path/to/sc_file.csv/'


In [ ]:
import pandas as pd 

scRNAseq = pd.read_csv(sc_file, header=None, index_col=0, dtype=object)
scRNAseq = scRNAseq.rename(columns=scRNAseq.iloc[0], copy=False).iloc[1:]
scRNAseq = scRNAseq.astype(float)


In [ ]:
# We can show the data to have a look and confirm everything looks as it should.
scRNAseq.head()

### Preprocessing the spots

Before running PCIseq, the ISS spots must be preprocessed and aligned with the scRNAseq dataset:

1. **Optional quality filtering**

   Spots can be filtered using decoding quality metrics (e.g. `quality_minimum`).

   * If `quality_threshold` is provided, only spots with  
     `quality_minimum > threshold` are kept.

   * If `None`, no filtering is applied and all decoded spots are used.

2. **Coordinate conversion**

   * If ISS spot coordinates are already in **pixels**, use  
     `pixel_to_um = 1`.

   * If coordinates are in **micrometers**, convert them into pixel units (to match the segmentation mask) using for example  
     `pixel_to_um = 0.1625`.

3. **Gene intersection**

   To ensure comparability, we restrict both ISS and scRNAseq data to the set of genes present in *both* modalities. Genes absent from one dataset are not informative for integration and are excluded.

#### Data flow

Here’s how the pieces fit together in the pipeline:

**Decoded ISS spots** (`spots_file`)  
⬇ filter (optional) + convert + gene intersection  

**Segmentation mask** (`segmentation_file`)  
⬇ assign spots → cells  

**scRNAseq reference** (`scRNAseq`)  
⬇ restrict to shared genes  

➡️ **Spatial gene-by-cell matrix** → input to **PCIseq**


#### Input Parameters

`spots_file` *(str or Path)*  Full path to the decoded ISS spots file (CSV).

`segmentation_file` *(str or Path)*  Full path to a segmentation mask (`.npz` file).  
The mask must be a sparse matrix where each pixel is labeled with a cell ID.

`scRNAseq` *(pd.DataFrame)*  Reference scRNAseq expression matrix (genes × cell types/clusters).

`quality_threshold` *(float or None, optional)*  Minimum decoding quality required for a spot to be included.  
If `None`, no filtering is applied.

`pixel_to_um` *(float, default=1.0)* Microns per pixel (µm/pixel) used to convert ISS spot coordinates into pixel units.

- If your spot coordinates are in **microns**, set this to your microscope pixel size  
  (e.g. `0.1625`), and coordinates will be converted to pixels.

- If your spot coordinates are already in **pixels**, use `pixel_to_um = 1.0`  
  (no conversion is applied).
  
`region` *(str, optional)*  Region label used for plotting and output organisation.

> **Note:** In this notebook, **only one region is processed at a time**.  
> The variable `region` is used for labelling and output structure, not for loading data from a fixed directory layout.

In [ ]:
coo_file = '/path/to/segmentation_file.csv/'
spots_file = '/path/to/spots_file.csv/'

output_dir_prefix = '/path/to/output_dir/'
region = 'R1'

In [ ]:
coo, spots, scrnaseq_clean = PCIseq.prepare_pciseq_inputs(
    spots_file=spots_file,
    coo_file=coo_file,
    scRNAseq=scRNAseq,   
    pixel_to_um=1,
    quality_threshold=0.5,
    shuffle_spots=True,
    random_state=42,
    plot=True,
    region="R1",
)

### Running PCIseq

Now we can finally run the **Probabilistic Cell Typing (PCIseq)** algorithm.  
This step integrates the ISS decoded spots with the segmentation mask and the 
reference scRNA-seq dataset to probabilistically assign each segmented cell to 
a transcriptional cell type.  

#### Input parameters

`region` *(str)* Region identifier (e.g., `"R1"`). Used for labelling and organising outputs.

`spots` DataFrame of ISS spots with columns `Gene`, `x`, `y` (pixel coordinates).

`coo_mask` Segmentation mask as a sparse matrix (cell labels per pixel).

`sc_expression_matrix` Reference scRNA-seq matrix (genes × cell types/clusters).

`output_dir_prefix` *(str or Path)*  Base output directory. Results are saved under:  
`<output_dir_prefix>/<region>/PCIseq/`

`save_output` *(optional)*  Whether to write results to disk (default: `True`).

`prob_threshold` *(float or None, optional)*  Minimum probability required for a cell to be included in the filtered output.  

- If `None`, no filtering is applied.  
- If provided, a filtered table is generated with only cells where  
  `Prob > prob_threshold` (e.g. `0.6`).

---

#### Output files

The following files are generated in the output directory:

- `cellData.json` — Cell-level assignments and probabilities  
- `geneData.json` — Gene-level contributions  
- `most_probable.csv` — Most probable cell type per cell  

If `prob_threshold` is provided:

- `most_probable_pXX.csv` — Filtered assignments (e.g. `most_probable_p06.csv`)

---

> **Note:** The current Python implementation can be quite slow, so please allow sufficient  
> time for the computation, especially when working with large datasets or many regions.

In [ ]:
cellData, geneData, most_probable, most_probable_filtered = PCIseq.run_pciseq(
    region="R1",
    spots=spots,
    coo_mask=coo,
    sc_expression_matrix=scrnaseq_clean,
    output_dir_prefix=output_dir_prefix,
    save_output=True,
    prob_threshold=0.6,
)

# Read the PCIseq output and plot the data

PCIseq is **probabilistic** in two important ways:  
1. It calculates the probability of each cell belonging to a specific cell type (or another).  
2. It calculates the probability of each ISS spot being assigned to a given cell (or another).  

After running PCIseq, the following three output files are generated in the region’s `postprocessing/PCIseq/` folder:  

- **`most_probable.csv`**  
  A table listing each segmented cell with its `(x, y)` position, the most probable assigned cell type, and the probability of that assignment.  

- **`geneData.json`**  
  A spot-level table showing which cell each spot has been assigned to, along with the assignment probability.  

- **`cellData.json`**  
  A more detailed cell-level table that includes secondary probabilities, assignment distributions across possible cell types, and other metadata.  

We begin by reading the **`most_probable.csv`** file.


In [ ]:
from pathlib import Path

PCIseq_dir = Path(output_dir_prefix) / region / "PCIseq"
pcifile = PCIseq_dir / "most_probable.csv"
pciout = pd.read_csv(pcifile)

The `pciout['ClassName']` column will contain the primary assignment for each cell, and the `pciout['Prob']` will contain the probability of that assignment.

For each cluster, we can plot the cells, together with their color-coded PCIseq probability using the following code:

In [ ]:
import matplotlib.pyplot as plt

# Loop over all unique assigned cell types
for cluster in pciout["ClassName"].dropna().unique():
    print(f"Plotting cluster: {cluster}")

    pcigene = pciout.loc[pciout["ClassName"] == cluster]

    plt.figure(figsize=(5, 5))  # smaller figure

    sc = plt.scatter(
        pcigene["X"],
        pcigene["Y"],
        c=pcigene["Prob"],
        s=3,          # also shrink point size
        alpha=0.8
    )

    plt.colorbar(sc, label="Probability", fraction=0.046, pad=0.04)
    plt.title(f"{cluster}", fontsize=10)
    plt.xlabel("X", fontsize=8)
    plt.ylabel("Y", fontsize=8)
    plt.axis("equal")

    plt.tight_layout()
    plt.show()